In [1]:
import polars as pl

### Load the OJPs datasets

In [2]:
ojp = pl.read_parquet("../data/lightcast/ITA_2025_postings.parquet")
#ojp_pd = pd.read_csv("../data/lightcast/ITA_2025_postings.csv")

# Only keep postings in Calabria (ITF6)
ojp_calabria_coded = ojp.filter(pl.col("LAA_ADMIN_AREA_1") == "ITF6")
del ojp

In [3]:
# Drop Undefined (99) and Volunteer (29) LOT Career Areas
ojp_calabria_coded = ojp_calabria_coded.filter(
    ~pl.col('LOT_CAREER_AREA').is_in(['99', '29'])
).unique()

### Decode OJP dataset using LightCast taxonomy dictionaries

In [4]:
ojp_calabria = ojp_calabria_coded.clone()

# Get all columns that match the LOT_* pattern
lot_columns = [col for col in ojp_calabria.columns if col.startswith(('LAA_', 'LOT_', 'NAICS_', 'ISCO_'))]

for col_name in lot_columns:
    # Auto-generate dict_name and name_col based on pattern
    dict_name = col_name.lower()  # LOT_CAREER_AREA -> lot_career_area
    name_col = f"{col_name}_NAME"  # LOT_CAREER_AREA -> LOT_CAREER_AREA_NAME
    code_col = f"{col_name}_CODE"  # LOT_CAREER_AREA -> LOT_CAREER_AREA_CODE
    
    # Load dictionary
    dict_df = pl.read_parquet(f"../data/lightcast/dictionaries/{dict_name}_dictionary.parquet")
    
    # Get original column order
    original_columns = ojp_calabria.columns
    col_position = original_columns.index(col_name)
    
    # Join to add the name column
    ojp_calabria = ojp_calabria.join(
        dict_df.select([col_name, name_col]),
        on=col_name,
        how='left'
    ).rename({col_name: code_col})
    
    # Reorder columns: insert CODE and NAME together at original position
    current_columns = ojp_calabria.columns
    other_columns = [c for c in current_columns if c not in [code_col, name_col]]
    new_order = other_columns[:col_position] + [code_col, name_col] + other_columns[col_position:]
    ojp_calabria = ojp_calabria.select(new_order)
    
    del dict_df

del new_order, lot_columns, original_columns, other_columns, current_columns
ojp_calabria = ojp_calabria.rename({col: col.lower() for col in ojp_calabria.columns}).unique("id")

### Map CP2021 (Training Catalogue) to ISCO-08 (LightCast)

In [5]:
cp2021_isco08_crosswalk = pl.read_csv("../data/isco08_cp2021_crosswalk.csv", schema_overrides={'isco_08_03_code': pl.String}, separator=";")

# Join cp2021_isco08_crosswalk to ojp_calabria on isco_08_3_code column
ojp_calabria = ojp_calabria.join(
    cp2021_isco08_crosswalk.select(['isco_08_03_code', 'cp2021_code', 'cp2021_name']),
    left_on='isco_08_3_code',
    right_on='isco_08_03_code',
    how='left'
)

### Load the skills datasets

In [6]:
skills_all = pl.read_parquet("../data/lightcast/ITA_2025_skills.parquet")
skills_dict = pl.read_parquet("../data/lightcast/dictionaries/skill_id_dictionary.parquet")

In [7]:
full_skill_list = pl.read_csv("../data/lightcast/dictionaries/full_skill_list.csv", separator=";")

In [12]:
skills_all = (skills_all.join(skills_dict, on='SKILL_ID', how='left')
          .select(['ID', 'SKILL_NAME', 'SKILL_ID'] + [col for col in skills_all.columns if col not in ['ID', 'SKILL_NAME', 'SKILL_ID']])).rename({col: col.lower() for col in skills_all.columns})
del skills_dict

In [13]:
# Get unique IDs from ojp_calabria
calabria_ids = ojp_calabria.select("id").unique()

# Filter skills_all lazily
skills_calabria = skills_all.lazy().join(
    calabria_ids.lazy(),
    on="id",
    how="inner"
).collect()

del skills_all
del calabria_ids

In [14]:
ojp_and_skills_calabria = skills_calabria.join(ojp_calabria, on='id', how='left')

In [8]:
jobs = pl.read_parquet("../data/lightcast/dictionaries/lot_occupation_dictionary.parquet")